exploration des méthodes possibles de calcul pour simuler les combats OGamiens

Rappel des règles de combats
- 6 tours max
- tir "simultanés" (mais y'a quand même un ordre)
- à chaque tir reçu, on applique la valeur d'attaque au bouclier (arrondi inf en % de la valeur du bouclier total) puis à la coque
    - de ce calcul arrondi découle la règle spéciale d'annulation des dégat si la valeur d'attaque vaut moins de 1% de la valeur des boucliers
- la coque = 10% des points de structure !!
- dès que la coque est endommagée à plus de 30%, le même % est appliqué (à chaque tir reçu) pour savoir si le vaisseau explose
- un vaisseau "détruit" (pendant le calcul d'un tour) n'est retiré qu'à la fin du tour et peut donc continuer à être ciblé ET tirer
- à la fin d'un tour, on retire les vaisseaux détruits et on recharge intégralement tous les boucliers
- à la fin des 6 tours s'il reste des vaisseaux/défenses dans les 2 camps => match nul

### Rapid fire
si un vaisseau tire sur une structure contre laquelle il a du rapid-fire, on lance un dé de la valeur du RF (eg croiseur RF 6 contre Ch lé). Il y a alors 1 chance sur RF (1/6) de ne PAS tirer à nouveau donc 1 - 1/RF d'effectuer un nouveau tir, le cycle continue jusqu'à ne plus proc le RF

## imports

In [1]:
import random
import numpy as np
import pandas as pd
from pathlib import Path

## lecture données

In [2]:
data_vaisseaux_path = Path("../data/vaisseaux.csv")
df_vaisseaux = pd.read_csv(data_vaisseaux_path)
df_vaisseaux

,nom,structure,bouclier,attaque,vitesse,fret,consommation
0,Petit transporteur,4000,10,5,10000,5000,20
1,Grand transporteur,12000,25,5,7500,25000,50
2,Chasseur léger,4000,10,50,12500,50,20
3,Chasseur lourd,10000,25,150,10000,100,75
4,Croiseur,27000,50,400,15000,800,300
5,Vaisseau de bataille,60000,200,1000,10000,1500,500
6,Bombardier,75000,500,1000,5000,500,700
7,Destructeur,110000,500,2000,5000,2000,1000
8,Etoile de la mort,9000000,50000,200000,100,1000000,1
9,Traqueur,70000,400,700,10000,750,250


pour convetir le csv en json "the lazy way"

In [9]:
df_vaisseaux.to_json(Path("../data/vaisseaux.json"), indent=4, force_ascii=False, orient="records")

## ajustements technologiques
lecture d'un... yaml ? ouais disons yaml qui contiendra toutes nos paramètres (univers, technologies)

Ajustera les stats des vaisseaux selon les formules

In [3]:
from yaml import safe_load

with Path("../data/universe_parameters.yaml").open("r") as f:
    universe_params = safe_load(f)

universe_params["Technologies"]

{'Réacteur à Combustion': 17,
 'Réacteur à Impultion': 12,
 'Propultion Hyperespace': 10,
 'Technologie Hyperespace': 12,
 'Technologie Armes': 17,
 'Technologie Bouclier': 16,
 'Technologie Protection des vaisseaux spatiaux': 17}

In [4]:
df_vaisseaux.attaque *= 1 + universe_params["Technologies"]["Technologie Armes"] / 10
df_vaisseaux.bouclier *= 1 + universe_params["Technologies"]["Technologie Bouclier"] / 10
df_vaisseaux.structure *= 1 + universe_params["Technologies"]["Technologie Protection des vaisseaux spatiaux"] / 10

In [5]:
df_vaisseaux

,nom,structure,bouclier,attaque,vitesse,fret,consommation
0,Petit transporteur,10800.0,26.0,13.5,10000,5000,20
1,Grand transporteur,32400.0,65.0,13.5,7500,25000,50
2,Chasseur léger,10800.0,26.0,135.0,12500,50,20
3,Chasseur lourd,27000.0,65.0,405.0,10000,100,75
4,Croiseur,72900.0,130.0,1080.0,15000,800,300
5,Vaisseau de bataille,162000.0,520.0,2700.0,10000,1500,500
6,Bombardier,202500.0,1300.0,2700.0,5000,500,700
7,Destructeur,297000.0,1300.0,5400.0,5000,2000,1000
8,Etoile de la mort,24300000.0,130000.0,540000.0,100,1000000,1
9,Traqueur,189000.0,1040.0,1890.0,10000,750,250


## création d'une flotte

l'idée serait de constituer un array numpy (on peut surement commencer par un df pandas...)  
afin de permettre ensuite le calcul vectoriel pour chaque individu de la flotte (le RF fera surement chier...)

In [6]:
from data_utils import replicate_rows

df_attaquants = replicate_rows(df_vaisseaux, [2000, 900, 0, 1100, 0, 75, 0, 25, 0, 35, 0, 450, 0, 0, 1, 0, 0])
df_defenseurs = replicate_rows(df_vaisseaux, [1500, 700, 0, 900, 0, 50, 0, 15, 0, 25, 0, 200, 0, 0, 0, 0, 0])

In [7]:
df_attaquants.reset_index(names="tireur", inplace=True)

In [8]:
df_attaquants['cible'] = np.random.randint(0, df_defenseurs.index.max()+1, size=len(df_attaquants))
df_attaquants

,tireur,nom,structure,bouclier,attaque,vitesse,fret,consommation,cible
0,0,Petit transporteur,10800.0,26.0,13.5,10000,5000,20,2102
1,1,Petit transporteur,10800.0,26.0,13.5,10000,5000,20,774
2,2,Petit transporteur,10800.0,26.0,13.5,10000,5000,20,35
3,3,Petit transporteur,10800.0,26.0,13.5,10000,5000,20,1857
4,4,Petit transporteur,10800.0,26.0,13.5,10000,5000,20,1757
...,...,...,...,...,...,...,...,...,...
4581,4581,Eclaireur,62100.0,260.0,540.0,12000,10000,300,222
4582,4582,Eclaireur,62100.0,260.0,540.0,12000,10000,300,2089
4583,4583,Eclaireur,62100.0,260.0,540.0,12000,10000,300,2259
4584,4584,Eclaireur,62100.0,260.0,540.0,12000,10000,300,229


In [9]:
df_defenseurs

,nom,structure,bouclier,attaque,vitesse,fret,consommation
0,Petit transporteur,10800.0,26.0,13.5,10000,5000,20
1,Petit transporteur,10800.0,26.0,13.5,10000,5000,20
2,Petit transporteur,10800.0,26.0,13.5,10000,5000,20
3,Petit transporteur,10800.0,26.0,13.5,10000,5000,20
4,Petit transporteur,10800.0,26.0,13.5,10000,5000,20
...,...,...,...,...,...,...,...
3385,Eclaireur,62100.0,260.0,540.0,12000,10000,300
3386,Eclaireur,62100.0,260.0,540.0,12000,10000,300
3387,Eclaireur,62100.0,260.0,540.0,12000,10000,300
3388,Eclaireur,62100.0,260.0,540.0,12000,10000,300


In [10]:
df_tirs = df_attaquants.merge(
    right=df_defenseurs, 
    how='left', 
    left_on='cible', 
    right_index=True,
    suffixes=["_A", "_D"]
)

### extrait du forum, sur le calcul des boucliers
https://board.fr.ogame.gameforge.com/index.php?thread/728567-casser-un-grand-bouclier/&postID=12026202#post12026202

Dans le cas ou le tir a une attaque inférieur au bouclier restant, le tir atteint le bouclier avec une valeur égale à l'arrondi au pourcent inférieur de la valeur maximale du bouclier (si c'est pas très clair, voir l'exemple ligne suivante qui sera plus parlant).

Dans ton cas, un croiseur attaque 720 sur un bouclier qui a une valeur max de 16000 : 720 / 16000 = 4.5%, arrondi inférieur = 4%, donc le bouclier va diminuer de 4% de sa valeur totale, soit 640.

Le tir de 720 est entièrement absorbé, mais ne diminue le bouclier que de 640.

C'est ce qui explique par ailleurs qu'un tir inférieur à 1% du bouclier ne tape pas du tout, vu qu'il est arrondi à 0% (et dans ton cas, les GT ne tapent donc pas le bouclier vu que c'est 0.06% du bouclier).

In [11]:
df_tirs["dmg_shield"] = (np.floor(100 * df_tirs.attaque_A / df_tirs.bouclier_D) * df_tirs.bouclier_D // 100).astype(int)
df_tirs = df_tirs.sort_values(by=["cible", "tireur"])[["tireur", "nom_A", "attaque_A", "cible", "nom_D", "bouclier_D", "structure_D", "dmg_shield"]]
# df_tirs

il y a un truc à faire avec un tri par cible puis attaquant, ce qui permet d'appliquer une somme cumulative conditionnelle ensuite et résoudre les tirs sur une opération simplifiée (somme et soustraction, pas de produit/division)

In [12]:
# dégats cumulés par cible dans l'ordre de l'id des tireurs
df_tirs["shield_aft_hit"] = df_tirs.bouclier_D - df_tirs.groupby("cible")["dmg_shield"].cumsum()

# colonne décalée pour avoir la valeur du shield avant le tir
df_tirs["shield_bef_hit"] = df_tirs.groupby("cible")["shield_aft_hit"].shift(1).fillna(df_tirs.bouclier_D).astype(int)

# vérifie si le tir dépasse la valeur max des boucliers
# ie. si les boucliers sont HS après le tir
# df_tirs["is_overkill"] = df_tirs.shield_aft_hit <= 0
is_overkill = df_tirs.shield_aft_hit <= 0 # on externalise le filtre booléen (pour alléger le df)

# vérifie si les boucliers étaient déjà HS avant le tir
# df_tirs["déjà_hs"] = df_tirs.shield_bef_hit <= 0
déjà_hs = df_tirs.shield_bef_hit <= 0

df_tirs

,tireur,nom_A,attaque_A,cible,nom_D,bouclier_D,structure_D,dmg_shield,shield_aft_hit,shield_bef_hit
2309,2309,Grand transporteur,13.5,0,Petit transporteur,26.0,10800.0,13,13.0,26
2618,2618,Grand transporteur,13.5,1,Petit transporteur,26.0,10800.0,13,13.0,26
1403,1403,Petit transporteur,13.5,2,Petit transporteur,26.0,10800.0,13,13.0,26
1534,1534,Petit transporteur,13.5,2,Petit transporteur,26.0,10800.0,13,0.0,13
3061,3061,Chasseur lourd,405.0,2,Petit transporteur,26.0,10800.0,404,-404.0,0
...,...,...,...,...,...,...,...,...,...,...
3442,3442,Chasseur lourd,405.0,3386,Eclaireur,260.0,62100.0,403,-143.0,260
3273,3273,Chasseur lourd,405.0,3388,Eclaireur,260.0,62100.0,403,-143.0,260
3345,3345,Chasseur lourd,405.0,3388,Eclaireur,260.0,62100.0,403,-546.0,-143
2361,2361,Grand transporteur,13.5,3389,Eclaireur,260.0,62100.0,13,247.0,260


### calcul des dégats infligés à la coque
le **breakpoint** est le moment où le tir est **overkill** mais les boucliers ne sont **pas** déjà HS avant le tir

In [13]:
# filtre (1ère passe) pour attribuer les dégats complets (ie. la totalité des dégats) infligés à la coque
hull_dmg = np.where(déjà_hs, df_tirs.attaque_A, 0)
hull_dmg

array([  0.,   0.,   0., ..., 405.,   0.,   0.], shape=(4586,))

In [14]:
# 2ème passe pour la situation de breakpoint
df_tirs["hull_dmg"] = np.where(
    (is_overkill) & (~déjà_hs), # filtre uniquement pour le breakpoint
    np.maximum(0, df_tirs.attaque_A - df_tirs.shield_bef_hit), # calcule la valeur d'attaque restante après les dégats aux boucliers
    hull_dmg
)
df_tirs["hull_dmg_cum"] = df_tirs.groupby("cible")["hull_dmg"].cumsum()
df_tirs

,tireur,nom_A,attaque_A,cible,nom_D,bouclier_D,structure_D,dmg_shield,shield_aft_hit,shield_bef_hit,hull_dmg,hull_dmg_cum
2309,2309,Grand transporteur,13.5,0,Petit transporteur,26.0,10800.0,13,13.0,26,0.0,0.0
2618,2618,Grand transporteur,13.5,1,Petit transporteur,26.0,10800.0,13,13.0,26,0.0,0.0
1403,1403,Petit transporteur,13.5,2,Petit transporteur,26.0,10800.0,13,13.0,26,0.0,0.0
1534,1534,Petit transporteur,13.5,2,Petit transporteur,26.0,10800.0,13,0.0,13,0.5,0.5
3061,3061,Chasseur lourd,405.0,2,Petit transporteur,26.0,10800.0,404,-404.0,0,405.0,405.5
...,...,...,...,...,...,...,...,...,...,...,...,...
3442,3442,Chasseur lourd,405.0,3386,Eclaireur,260.0,62100.0,403,-143.0,260,145.0,145.0
3273,3273,Chasseur lourd,405.0,3388,Eclaireur,260.0,62100.0,403,-143.0,260,145.0,145.0
3345,3345,Chasseur lourd,405.0,3388,Eclaireur,260.0,62100.0,403,-546.0,-143,405.0,550.0
2361,2361,Grand transporteur,13.5,3389,Eclaireur,260.0,62100.0,13,247.0,260,0.0,0.0


calcul du % d'endommagement de la coque  
Rappels : 
+ au delà de 30% de dégats, % de destruction = % dégats
+ coque = 10% des points de structure (= coût M + coût C, btw)


In [15]:
hull_dmg_percent = np.minimum(1, 10 * df_tirs.hull_dmg_cum / df_tirs.structure_D)
df_tirs["destruction_chance"] = np.where(
    hull_dmg_percent > 0.3,
    hull_dmg_percent,
    0
)
destruction_roll = np.random.random(size=len(df_tirs))
df_tirs["is_destroyed"] = destruction_roll < df_tirs["destruction_chance"]
df_tirs

,tireur,nom_A,attaque_A,cible,nom_D,bouclier_D,structure_D,dmg_shield,shield_aft_hit,shield_bef_hit,hull_dmg,hull_dmg_cum,destruction_chance,is_destroyed
2309,2309,Grand transporteur,13.5,0,Petit transporteur,26.0,10800.0,13,13.0,26,0.0,0.0,0.000000,False
2618,2618,Grand transporteur,13.5,1,Petit transporteur,26.0,10800.0,13,13.0,26,0.0,0.0,0.000000,False
1403,1403,Petit transporteur,13.5,2,Petit transporteur,26.0,10800.0,13,13.0,26,0.0,0.0,0.000000,False
1534,1534,Petit transporteur,13.5,2,Petit transporteur,26.0,10800.0,13,0.0,13,0.5,0.5,0.000000,False
3061,3061,Chasseur lourd,405.0,2,Petit transporteur,26.0,10800.0,404,-404.0,0,405.0,405.5,0.375463,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3442,3442,Chasseur lourd,405.0,3386,Eclaireur,260.0,62100.0,403,-143.0,260,145.0,145.0,0.000000,False
3273,3273,Chasseur lourd,405.0,3388,Eclaireur,260.0,62100.0,403,-143.0,260,145.0,145.0,0.000000,False
3345,3345,Chasseur lourd,405.0,3388,Eclaireur,260.0,62100.0,403,-546.0,-143,405.0,550.0,0.000000,False
2361,2361,Grand transporteur,13.5,3389,Eclaireur,260.0,62100.0,13,247.0,260,0.0,0.0,0.000000,False


In [16]:
df_morts = df_tirs[df_tirs.is_destroyed][["cible", "nom_D", "structure_D", "hull_dmg_cum", "destruction_chance"]]
df_morts

,cible,nom_D,structure_D,hull_dmg_cum,destruction_chance
3708,7,Petit transporteur,10800.0,784.0,0.725926
4376,7,Petit transporteur,10800.0,1324.0,1.000000
4302,10,Petit transporteur,10800.0,540.5,0.500463
4006,11,Petit transporteur,10800.0,3079.0,1.000000
3256,12,Petit transporteur,10800.0,392.0,0.362963
...,...,...,...,...,...
4070,3264,Eclaireur,62100.0,2845.0,0.458132
4097,3290,Eclaireur,62100.0,5179.0,0.833977
4380,3290,Eclaireur,62100.0,5719.0,0.920934
4080,3314,Eclaireur,62100.0,5140.0,0.827697
